In [0]:
# ---- imports.py ----
import requests
import os

SARVAM_API_KEY = dbutils.secrets.get(scope="api_keys_scope", key="sarvam_api_key")
os.environ["SARVAM_API_KEY"] = SARVAM_API_KEY

print("✅ Translator imports done")

✅ Translator imports done


In [0]:
# ---- detect_language.py ----
def detect_language(text):
    url     = "https://api.sarvam.ai/text-lid"
    headers = {
        "api-subscription-key": SARVAM_API_KEY,
        "Content-Type"        : "application/json"
    }
    payload  = {"input": text}
    response = requests.post(url, json=payload, headers=headers)
    
    if response.status_code == 200:
        lang = response.json().get("language_code", "en-IN")
        print(f"✅ Detected language: {lang}")
        return lang
    else:
        print(f"⚠️ Detection failed: {response.text} — defaulting to en-IN")
        return "en-IN"

# Test
detected = detect_language("मुझे बहरीन को निर्यात करने के लिए क्या दस्तावेज चाहिए?")
print(f"Detected: {detected}")

✅ Detected language: hi-IN
Detected: hi-IN


In [0]:
# ---- translate_to_english.py ----
def translate_to_english(text, source_lang="hi-IN"):
    if source_lang.startswith("en"):
        return text  # already english, skip API call
    
    url     = "https://api.sarvam.ai/translate"
    headers = {
        "api-subscription-key": SARVAM_API_KEY,
        "Content-Type"        : "application/json"
    }
    payload = {
        "input"                : text,
        "source_language_code" : source_lang,
        "target_language_code" : "en-IN",
        "model"                : "mayura:v1",
        "enable_preprocessing" : True
    }
    response = requests.post(url, json=payload, headers=headers)

    if response.status_code == 200:
        translated = response.json().get("translated_text", text)
        print(f"✅ Translated to English: {translated}")
        return translated
    else:
        print(f"⚠️ Translation failed: {response.text} — returning original")
        return text

# Test
english = translate_to_english(
    "मुझे बहरीन को निर्यात करने के लिए क्या दस्तावेज चाहिए?",
    source_lang="hi-IN"
)
print(f"English: {english}")

✅ Translated to English: What documents do I need for exporting to Bahrain?
English: What documents do I need for exporting to Bahrain?


In [0]:
# ---- translate_to_user_lang.py ----
def translate_to_user_lang(text, target_lang="hi-IN"):
    if target_lang.startswith("en"):
        return text  # no translation needed

    url     = "https://api.sarvam.ai/translate"
    headers = {
        "api-subscription-key": SARVAM_API_KEY,
        "Content-Type"        : "application/json"
    }
    payload = {
        "input"                : text,
        "source_language_code" : "en-IN",
        "target_language_code" : target_lang,
        "model"                : "mayura:v1",
        "enable_preprocessing" : True
    }
    response = requests.post(url, json=payload, headers=headers)

    if response.status_code == 200:
        translated = response.json().get("translated_text", text)
        print(f"✅ Translated to {target_lang}")
        return translated
    else:
        print(f"⚠️ Translation failed: {response.text} — returning original")
        return text

# Test
hindi = translate_to_user_lang(
    "You need a phytosanitary certificate to export to Bahrain.",
    target_lang="hi-IN"
)
print(f"Hindi: {hindi}")

✅ Translated to hi-IN
Hindi: बहरीन को निर्यात करने के लिए आपको एक पादप स्वास्थ्य प्रमाणपत्र की आवश्यकता है।


In [0]:
# ---- translation_pipeline.py ----
def translation_pipeline(user_text):
    """
    Full pipeline:
    1. Detect language
    2. Translate to English
    Returns (english_text, detected_lang)
    """
    detected_lang  = detect_language(user_text)
    english_text   = translate_to_english(user_text, source_lang=detected_lang)
    return english_text, detected_lang

# Test full pipeline
english_query, user_lang = translation_pipeline(
    "மலேசியாவிற்கு ஏற்றுமதி செய்ய என்ன ஆவணங்கள் தேவை?"  # Tamil
)
print(f"\nOriginal language : {user_lang}")
print(f"English query     : {english_query}")

✅ Detected language: ta-IN
✅ Translated to English: What documents are required for export to Malaysia?

Original language : ta-IN
English query     : What documents are required for export to Malaysia?
